## Phase 2: Build Multivariate Time-Series Dataset

We'll create two datasets:
 - **panel_quarterly_org.csv**: one row per (organization_id, quarter)
 - **panel_annual_org.csv** : one row per (organization_id, year)

Each includes:
 - **funding** target
 - **topic prevalence** (sum of weights per topic)
 - **deliverable_count**, **publication_count**
 - static org features (country, SME flag, etc.)

In [3]:
# 1) Imports & paths
import pandas as pd
from pathlib import Path
import sys

project_root = Path.cwd().parent  # assumes you're in /notebooks
sys.path.append(str(project_root))

           # adjust to your repo root
data_dir     = project_root / 'data' / 'processed'
agg_dir      = project_root / 'data' / 'aggregated'


# %%
# 2) Load processed tables
#   – projects.csv: for funding & start_date
#   – project_organizations.csv: maps project→organization
#   – project_topic_weights.csv: from Phase 1
#   – deliverables.csv / publications.csv: for counts
#   – organizations.csv: static covariates

projects       = pd.read_csv(data_dir / 'projects.csv', parse_dates=['start_date'])
proj_org       = pd.read_csv(data_dir / 'project_organizations.csv')
ptw_wide       = pd.read_csv(agg_dir  / 'project_topic_weights.csv')
deliverables   = pd.read_csv(data_dir / 'deliverables.csv')
publications   = pd.read_csv(data_dir / 'publications.csv')
organizations  = pd.read_csv(data_dir / 'organizations.csv')

# **Fix datetime parsing for content_update_date**
deliverables['content_update_date'] = pd.to_datetime(
    deliverables['content_update_date'],
    errors='coerce'
)
publications['content_update_date'] = pd.to_datetime(
    publications['content_update_date'],
    errors='coerce'
)

deliverables = deliverables.dropna(subset=['content_update_date'])
publications = publications.dropna(subset=['content_update_date'])

# 3) Enforce consistent dtypes for project_id
for df in (proj_org, ptw_wide, deliverables, publications):
    df['project_id'] = pd.to_numeric(df['project_id'], errors='coerce').astype('Int64')
    df.dropna(subset=['project_id'], inplace=True)
    df['project_id'] = df['project_id'].astype(int)
    
# drop any rows where conversion failed (if any)
proj_org       = proj_org.dropna(subset=['project_id']).astype({'project_id':'int64'})
ptw_wide       = ptw_wide.dropna(subset=['project_id']).astype({'project_id':'int64'})
deliverables   = deliverables.dropna(subset=['project_id']).astype({'project_id':'int64'})
publications   = publications.dropna(subset=['project_id']).astype({'project_id':'int64'})

projects.name       = 'projects'
proj_org.name       = 'project_organizations'
ptw_wide.name       = 'project_topic_weights'
deliverables.name   = 'deliverables'
publications.name   = 'publications'
organizations.name  = 'organizations'
# %%


# print columns of all dataframes : like DF name: col1, col2, ...
for df in [projects, proj_org, ptw_wide, deliverables, publications, organizations]:
    print(f"{df.name}: {', '.join(df.columns)}")

projects: id, acronym, status, title, start_date, end_date, total_cost, ec_max_contribution, ec_signature_date, framework_programme, master_call, sub_call, funding_scheme, nature, objective, content_update_date, rcn, grant_doi, duration_days, duration_months, duration_years, n_institutions, coordinator_name, ec_contribution_per_year, total_cost_per_year, field_class, field, sub_field, niche
project_organizations: project_id, organization_id, role, order_index, ec_contribution, net_ec_contribution, total_cost, end_of_participation, active
project_topic_weights: project_id, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 1

In [4]:
# 3) Build the flat panel
#    a) Start from projects, rename id→project_id
projects = projects.rename(columns={'id':'project_id'})

panel = projects.copy()

#    b) Merge topic weights (one row per project)
panel = panel.merge(ptw_wide, on='project_id', how='left')

#    c) Add deliverable & publication counts
deliv = deliverables.groupby('project_id').size().rename('deliverable_count')
pub   = publications.groupby('project_id').size().rename('publication_count')
panel = panel.merge(deliv, on='project_id', how='left')
panel = panel.merge(pub,   on='project_id', how='left')
panel[['deliverable_count','publication_count']] = panel[['deliverable_count','publication_count']].fillna(0)


# d) EC funding target: use net_ec_contribution
#    Sum across all partners (or filter role=='coordinator' if you want only that share)
fund = (
    proj_org
    .groupby('project_id')['net_ec_contribution']
    .sum()
    .rename('funding')
)
panel = panel.merge(fund, on='project_id', how='left')
panel = panel.drop(columns=['ec_max_contribution'], errors='ignore')

# e) Add coordinator’s static org features
coord = (
    proj_org[proj_org.role.str.lower()=='coordinator']
    .merge(organizations, left_on='organization_id', right_on='id', suffixes=('','_org'))
    [['project_id','country','sme','activity_type']]
    .rename(columns={
        'country':'coord_country',
        'sme':'coord_sme',
        'activity_type':'coord_activity_type'
    })
)
panel = panel.merge(coord, on='project_id', how='left')
panel[['coord_country','coord_sme','coord_activity_type']] = panel[['coord_country','coord_sme','coord_activity_type']].fillna('unknown')



In [5]:
# 4) Clean & filter
#    a) Drop duplicates (should be none, but just in case)
panel = panel.drop_duplicates(subset=['project_id'])

#    b) Drop rows with no funding
panel = panel[panel['funding'].notnull() & (panel['funding']>0)]

#    c) Drop non-informative columns
#       (any with only one unique value or all-missing)
drop_cols = [c for c in panel.columns 
             if panel[c].nunique(dropna=True)<=1 or panel[c].isnull().all()]
panel = panel.drop(columns=drop_cols)

# 5) Quick check
print(f"Projects: {panel['project_id'].nunique()} rows: {panel.shape[0]}")
print("Missing in target:", panel['funding'].isnull().sum())
print("Any duplicates left?", panel['project_id'].duplicated().any())

# 6) Save final modelling set
panel.to_csv(agg_dir/'modeling_panel_project.csv', index=False)
print("✅ Saved modelling panel to modeling_panel_project.csv")

Projects: 15863 rows: 15863
Missing in target: 0
Any duplicates left? False
✅ Saved modelling panel to modeling_panel_project.csv


In [10]:
# ─── Load your already‐built panel ─────────────────────────────────────────────
panel = pd.read_csv(agg_dir / 'modeling_panel_project.csv',
                    parse_dates=['ec_signature_date'])

# Confirm we have the right columns
print("Available columns:", panel.columns.tolist())

# ─── 1) Keep only signed projects ───────────────────────────────────────────────
panel = panel[panel['status'].str.upper() == 'SIGNED']

# ─── 2) Filter by signature date ────────────────────────────────────────────────
cutoff = pd.to_datetime('2025-05-26')
# Ensure ec_signature_date is datetime:
panel['ec_signature_date'] = pd.to_datetime(panel['ec_signature_date'], errors='coerce')
panel = panel[panel['ec_signature_date'] <= cutoff]

# ─── 3) Drop any zero or missing funding ────────────────────────────────────────
# Our target column is 'funding'
print("Before filtering funded:", panel.shape)
panel = panel[panel['funding'].notnull() & (panel['funding'] > 0)]
print("After filtering funded :", panel.shape)

# ─── Quick sanity checks ───────────────────────────────────────────────────────
print("Target distribution:\n", panel['funding'].describe())

# If you still see ~8k rows missing, check how many were removed at each step:
total   = pd.read_csv(agg_dir / 'modeling_panel_project.csv').shape[0]
signed  = pd.read_csv(agg_dir / 'modeling_panel_project.csv').query("status=='SIGNED'").shape[0]
print(f"Total rows: {total}, Signed: {signed}, Signed+funded: {panel.shape[0]}")


# How many had no ec_signature_date?
print("No signature date:", panel['ec_signature_date'].isna().sum())
# How many had funding==0?
orig = pd.read_csv(agg_dir/'modeling_panel_project.csv')
print("Zero funding rows:", (orig['funding']==0).sum())


# ─── 4) Save the filtered dataset ──────────────────────────────────────────────
panel.to_csv(agg_dir/'modeling_panel_project.csv', index=False)
print("✅ Saved:", agg_dir/'modeling_panel_project.csv')

Available columns: ['project_id', 'acronym', 'status', 'title', 'start_date', 'end_date', 'total_cost', 'ec_signature_date', 'master_call', 'sub_call', 'funding_scheme', 'objective', 'content_update_date', 'rcn', 'grant_doi', 'duration_days', 'duration_months', 'duration_years', 'n_institutions', 'coordinator_name', 'ec_contribution_per_year', 'total_cost_per_year', 'field_class', 'field', 'sub_field', 'niche', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '

In [12]:
# Calculate how many project we have
num_projects = panel['project_id'].nunique()
print(f"Total number of projects in the panel: {num_projects}")

print(panel.shape)
print(panel.columns.tolist())
print(panel.head())

# Count duplicates by project_id
dup_counts = panel['project_id'].value_counts()
print("Projects with duplicates (top 10):")
print(dup_counts[dup_counts > 1].head(10))

# Optionally, remove duplicates
panel_proj_nodup = panel.drop_duplicates(subset=['project_id'])

# If target is ec_max_contribution or net_ec_contribution, check its distribution
# Figure out which column is actually in the panel
if 'funding' in panel.columns:
    target = 'funding'
elif 'ec_max_contribution' in panel.columns:
    target = 'ec_max_contribution'
elif 'net_ec_contribution' in panel.columns:
    target = 'net_ec_contribution'
else:
    raise KeyError("No known funding column found!")

print(f"Analyzing target: {target}")
print(panel[target].describe())
print("Missing or zero funding:", (panel[target].isna() | (panel[target] == 0)).sum())
print("Proportion missing/zero:", ((panel[target].isna() | (panel[target] == 0)).mean()))


print(panel[target].describe())
print("Number of projects with missing/zero funding:", (panel[target].isna() | (panel[target]==0)).sum())
print("Proportion missing or zero funding:", (panel[target].isna() | (panel[target]==0)).mean())

missing = panel.isnull().mean().sort_values(ascending=False)
print("Columns with most missing values:")
print(missing.head(15))


for col in panel.columns:
    nunique = panel[col].nunique(dropna=True)
    all_zero = (panel[col]==0).all() if panel[col].dtype != object else False
    if nunique <= 1 or all_zero:
        print(f"Non-informative column: {col} (unique values: {nunique}, all zero: {all_zero})")
        

topic_cols = [str(i) for i in range(27) if str(i) in panel.columns]
for col in topic_cols:
    nonzero = (panel[col] > 0).sum()
    print(f"Topic {col}: nonzero in {nonzero} of {len(panel)} projects")


print(panel.dtypes.value_counts())
print(panel.corr(numeric_only=True).abs().sort_values(target, ascending=False)[target].head(10))




Total number of projects in the panel: 14819
(14819, 344)
['project_id', 'acronym', 'status', 'title', 'start_date', 'end_date', 'total_cost', 'ec_signature_date', 'master_call', 'sub_call', 'funding_scheme', 'objective', 'content_update_date', 'rcn', 'grant_doi', 'duration_days', 'duration_months', 'duration_years', 'n_institutions', 'coordinator_name', 'ec_contribution_per_year', 'total_cost_per_year', 'field_class', 'field', 'sub_field', 'niche', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92'